# 👑 QUILLAN-RONIN v5.4-ONI — INTERACTIVE MASTER TRAINING WORKBENCH (FIXED)
=============================================================================
**Unified Sovereign Architecture | BitNet 1.58b STE | MoBA Long-Context | MuonK2 Optimizer**

### FIXES APPLIED (2026-09-09 Antigravity Patch)
- Fixed `REPO_ROOT` path corruption (`C:\u0002_QUILLAN` -> `C:\02_QUILLAN`)
- Added CUDA CC 6.1 (GTX 1050) incompatibility fallback -> forced CPU Zero-Lag mode
- Fixed training loop `idx` bug: same sample repeated 8x per step -> now random/shuffled sampling
- Added `random` shuffle per epoch, proper effective batch scaling
- Fixed checkpoint resume (`start_step` off-by-one) and speed calc
- Added validation split + loss plotting
- Wrapped `model.generate` with device-safe helper

This notebook provides an interactive training and evaluation workbench directly inside the IDE.
- Live in-cell loss tracking and progress monitoring
- High-Density Anchor Curriculum to drive loss down to **sub-3.0**
- Real-time generation probes and interactive prompt testing


In [ ]:
# [Cell 1] Environment, Hardware Governor & Auto-Detect (Colab / Local) - FIXED
import os
import sys
import time
import math
import json
import random
import logging
from pathlib import Path
from typing import List, Tuple, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F

# Auto-detect Google Colab vs Local IDE Environment
try:
    import google.colab
    IN_COLAB = True
    print("✓ Running in Google Colab Cloud GPU environment!")
    REPO_ROOT = Path("/content/Quillan-Ronin")
except ImportError:
    IN_COLAB = False
    print("✓ Running locally inside IDE (Local Windows Mode)!")
    REPO_ROOT = Path(r"C:\02_QUILLAN")  # FIXED: was C:\u0002_QUILLAN (unicode escape)

for p in [
    REPO_ROOT / "09 - Projects" / "projects" / "oni",
    REPO_ROOT / "03 - Training & Model" / "scripts",
    REPO_ROOT / "scripts",
    REPO_ROOT,
]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

# Hardware Governor (Ensures 100% desktop & UI responsiveness on local CPU)
if not torch.cuda.is_available():
    torch.set_num_threads(min(2, os.cpu_count() or 2))
    torch.set_num_interop_threads(min(2, os.cpu_count() or 2))
try:
    import psutil
    p = psutil.Process()
    if hasattr(psutil, "BELOW_NORMAL_PRIORITY_CLASS"):
        p.nice(psutil.BELOW_NORMAL_PRIORITY_CLASS)
except Exception:
    pass

# Device selection with SM61 fallback (GTX 1050 not supported by PyTorch 2.8+ cu12.6)
if torch.cuda.is_available():
    try:
        major, minor = torch.cuda.get_device_capability(0)
        cc = major * 10 + minor
        if major < 7 or (major == 7 and minor < 5):
            print(f"⚠️ CUDA device CC {major}.{minor} not supported by this PyTorch build -> forcing CPU")
            device = torch.device("cpu")
        else:
            device = torch.device("cuda")
    except Exception as e:
        print(f"⚠️ CUDA check failed ({e}) -> forcing CPU")
        device = torch.device("cpu")
else:
    device = torch.device("cpu")

if device.type == "cpu":
    torch.set_num_threads(min(2, os.cpu_count() or 2))

print(f"✓ PyTorch version: {torch.__version__}")
try:
    dev_desc = torch.cuda.get_device_name(0) if device.type=="cuda" else f"CPU ({torch.get_num_threads()} threads, Zero-Lag Mode)"
except Exception:
    dev_desc = "CPU (Zero-Lag Mode)"
print(f"✓ Active Device: {device} [{dev_desc}]")
print(f"✓ REPO_ROOT resolved: {REPO_ROOT} (exists={REPO_ROOT.exists()})")


In [ ]:
# [Cell 2] Load Canonical Architecture & Checkpoint - FIXED
from quillan_v5_4_oni import QuillanRoninOni, QuillanOniConfig
from sovereign_inference_engine import SovereignTokenizer
from quillan_muonk2_optimizer import create_quillan_muonk2_optimizer

tokenizer = SovereignTokenizer("gpt2")
print(f"Tokenizer vocab: {tokenizer.vocab_size}, eos={tokenizer.eos_token_id}")

cfg = QuillanOniConfig(
    n_layer=6,
    hidden_dim=1024,
    max_seq_len=256,
    num_experts=34,
    router_mode="dense_pull",
    use_moba=True,
    moba_block_size=64,
    moba_top_k=2,
)

print("[1/2] Instantiating Quillan-Ronin v5.4-ONI architecture...")
model = QuillanRoninOni(cfg).to(device)
print(f"  -> params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

# Discover latest/best checkpoint
CKPT_DIR = REPO_ROOT / "checkpoints" / "checkpoints_sft"
CKPT_DIR.mkdir(parents=True, exist_ok=True)
candidates = [
    CKPT_DIR / "quillan_frontier_v2_best.pt",
    CKPT_DIR / "quillan_frontier_v2_latest.pt",
    REPO_ROOT / "checkpoints" / "production_export" / "quillan_ronin_v531_sovereign_production.pt",
    REPO_ROOT / "checkpoints" / "checkpoints_oni" / "quillan_oni_weights.pt",
]

loaded_ckpt = None
start_step = 1
best_loss = float("inf")

for c in candidates:
    if c.exists():
        print(f"[2/2] Loading checkpoint weights from: {c.name} ({c.stat().st_size/1e6:.1f} MB)")
        ckpt_data = torch.load(str(c), map_location=device, weights_only=False)
        sd = ckpt_data.get("model_state_dict", ckpt_data.get("model", ckpt_data))
        missing, unexpected = model.load_state_dict(sd, strict=False)
        print(f"✓ Checkpoint loaded: {len(sd) - len(unexpected)} matching tensors, {len(missing)} missing, {len(unexpected)} unexpected.")
        start_step = int(ckpt_data.get("step", 1)) + 1  # FIXED: resume at next step
        best_loss = float(ckpt_data.get("loss", float("inf")))
        loaded_ckpt = c
        break

if loaded_ckpt is None:
    print("⚠️ No checkpoint found - training from scratch")
else:
    print(f"✓ Model ready! Resuming at Step {start_step}, Best Recorded Loss: {best_loss:.4f}")


In [ ]:
# [Cell 3] High-Density Anchor Curriculum (Fast Sub-3.0 Convergence)
DATA_DIR = REPO_ROOT / "training_data"
MAX_SEQ_LEN = 256

def encode_qa_sample(prompt: str, response: str) -> Optional[Tuple[torch.Tensor, torch.Tensor]]:
    p_ids = tokenizer.encode(f"<|user|>\n{prompt.strip()}\n<|assistant|>\n")
    r_ids = tokenizer.encode(f"{response.strip()}<|im_end|>")
    seq = p_ids + r_ids
    labels = [-100] * len(p_ids) + list(r_ids)
    if len(seq) > MAX_SEQ_LEN:
        if MAX_SEQ_LEN - len(p_ids) < 15:
            return None
        seq = seq[:MAX_SEQ_LEN]
        labels = labels[:MAX_SEQ_LEN]
    pad = MAX_SEQ_LEN - len(seq)
    # FIXED: use tokenizer.eos_token_id (0 or 50256) consistently
    pad_id = tokenizer.eos_token_id if hasattr(tokenizer, 'eos_token_id') else 50256
    inp = seq + [pad_id] * pad
    lbl = labels + [-100] * pad
    if sum(1 for l in lbl if l != -100) < 5:
        return None
    return torch.tensor(inp, dtype=torch.long), torch.tensor(lbl, dtype=torch.long)

# Load Curated High-Density Anchor Datasets
anchors = []

# 1. Direct Factual & Syllogism Anchors
direct_path = DATA_DIR / "Quillan_Direct_Answers_Gold.jsonl"
if direct_path.exists():
    with open(direct_path, "r", encoding="utf-8") as f:
        for line in f:
            d = json.loads(line)
            pair = encode_qa_sample(d.get("prompt", ""), d.get("response", ""))
            if pair: anchors.append(pair)
    print(f"  Direct anchors: {len(anchors)}")

# 2. Sovereign Thinking Gold (<think> reasoning chains)
think_path = DATA_DIR / "sovereign_thinking_gold.jsonl"
prev = len(anchors)
if think_path.exists():
    with open(think_path, "r", encoding="utf-8") as f:
        for line in f:
            d = json.loads(line)
            pair = encode_qa_sample(d.get("prompt", ""), d.get("response", ""))
            if pair: anchors.append(pair)
    print(f"  Thinking anchors: {len(anchors)-prev}")

# 3. Clean Reasoning Gold
clean_path = DATA_DIR / "Quillan_Clean_Reasoning_Gold_Dataset.jsonl"
prev = len(anchors)
if clean_path.exists():
    with open(clean_path, "r", encoding="utf-8") as f:
        for line in f:
            d = json.loads(line)
            pair = encode_qa_sample(d.get("question", ""), d.get("response", ""))
            if pair: anchors.append(pair)
    print(f"  Clean reasoning: {len(anchors)-prev}")

print(f"✓ Loaded {len(anchors)} High-Density Anchor samples for rapid convergence!")
if len(anchors) == 0:
    raise RuntimeError("No anchors loaded - check training_data paths")

# Shuffle once + split 5% validation for monitoring
random.shuffle(anchors)
n_val = max(1, int(len(anchors)*0.05))
val_anchors = anchors[:n_val]
train_anchors = anchors[n_val:]
print(f"  -> train: {len(train_anchors)}, val: {len(val_anchors)}")


In [ ]:
# [Cell 4] Interactive Training Loop with Live In-Cell Loss Tracking - FIXED
import random
TRAIN_STEPS = 200          # Increase to 500-1000 per chunk on CPU; 200 is ~3-5 min on 2 threads
BATCH_SIZE = 2             # Batch size
ACCUM_STEPS = 4            # Gradient accumulation (EffBatch = 8)
LEARNING_RATE = 0.012      # Muon learning rate (AdamW side is 1e-4 inside optimizer)

optimizer = create_quillan_muonk2_optimizer(
    model,
    lr_muon=LEARNING_RATE,
    lr_adamw=1e-4,
    weight_decay=0.01,
    ccrl_limit=4.0,
)

model.train()
print(f"Starting Interactive Training: {TRAIN_STEPS} steps (EffBatch={BATCH_SIZE*ACCUM_STEPS}, train_pool={len(train_anchors)})...")
print("-" * 75)

step_losses = []
t0 = time.time()
N_train = len(train_anchors)
# FIXED: global sample counter for proper shuffling without repetition
global_sample_idx = 0
indices = list(range(N_train))
random.shuffle(indices)

def get_batch():
    global global_sample_idx, indices
    batch_inp, batch_lbl = [], []
    for _ in range(BATCH_SIZE):
        if global_sample_idx >= N_train:
            random.shuffle(indices)
            global_sample_idx = 0
        idx = indices[global_sample_idx]
        global_sample_idx += 1
        inp, lbl = train_anchors[idx]
        batch_inp.append(inp)
        batch_lbl.append(lbl)
    return torch.stack(batch_inp).to(device), torch.stack(batch_lbl).to(device)

for step in range(start_step, start_step + TRAIN_STEPS):
    optimizer.zero_grad(set_to_none=True)
    accum_loss = 0.0
    
    for _ in range(ACCUM_STEPS):
        inp_t, lbl_t = get_batch()
        out = model(inp_t, labels=lbl_t, return_aux=False)
        # model returns (logits, loss) when return_aux=False
        if isinstance(out, tuple) and len(out)==2:
            logits, loss = out
        else:
            logits, loss = out[0], out[1]
        (loss / ACCUM_STEPS).backward()
        accum_loss += loss.item() / ACCUM_STEPS

    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    
    step_losses.append(accum_loss)
    dt = time.time() - t0
    speed = 1.0 / max(dt, 1e-4)
    t0 = time.time()
    
    if accum_loss < best_loss:
        best_loss = accum_loss
        save_path = CKPT_DIR / "quillan_frontier_v2_best.pt"
        torch.save({"model_state_dict": model.state_dict(), "step": step, "loss": best_loss}, str(save_path))
        print(f"  ★ Step [{step:4d}] | Loss: {accum_loss:.4f} | 🏆 NEW BEST! | {speed:.2f} step/s | saved best")
    elif step % 10 == 0 or step == start_step:
        print(f"  • Step [{step:4d}] | Loss: {accum_loss:.4f} | {speed:.2f} step/s")

# Save latest (FIXED: step count is last executed step +1)
latest_path = CKPT_DIR / "quillan_frontier_v2_latest.pt"
torch.save({"model_state_dict": model.state_dict(), "step": start_step + TRAIN_STEPS -1, "loss": step_losses[-1]}, str(latest_path))
print("-" * 75)
print(f"✓ Training block completed! Final Step Loss: {step_losses[-1]:.4f}, Best Loss: {best_loss:.4f}")
print(f"  Latest saved to: {latest_path}")
# Quick val check
model.eval()
with torch.no_grad():
    val_losses=[]
    for i in range(min(20, len(val_anchors))):
        inp,lbl = val_anchors[i]
        out = model(inp.unsqueeze(0).to(device), labels=lbl.unsqueeze(0).to(device), return_aux=False)
        _, vloss = out if isinstance(out,tuple) and len(out)==2 else (out[0], out[1])
        val_losses.append(vloss.item())
    print(f"  Val loss (20 samples): {sum(val_losses)/len(val_losses):.4f}")
model.train()


In [ ]:
# [Cell 5] Interactive Prompt Testing & Generation Probe - FIXED
import torch
@torch.no_grad()
def ask_quillan(prompt: str, max_tokens: int = 150, temperature: float = 0.7) -> str:
    model.eval()
    formatted = f"<|user|>\n{prompt.strip()}\n<|assistant|>\n"
    tokens = tokenizer.encode(formatted)
    # Use the model's native generate (quillan_v5_4_oni.QuillanRoninOni.generate)
    try:
        gen_tokens = model.generate(
            tokens,
            max_tokens=max_tokens,
            temp=temperature,
            top_k=40,
            top_p=0.90,
        )
        # generate may return list[int] or tensor
        if isinstance(gen_tokens, torch.Tensor):
            gen_tokens = gen_tokens.tolist()
    except Exception as e:
        print(f"generate() failed: {e} - falling back to manual loop")
        # Fallback: manual autoregressive
        gen_tokens = list(tokens)
        for _ in range(max_tokens):
            inp = torch.tensor([gen_tokens[-256:]], dtype=torch.long, device=device)
            logits = model(inp)
            if isinstance(logits, tuple): logits = logits[0]
            next_logits = logits[0, -1, :] / max(temperature, 1e-4)
            probs = torch.softmax(next_logits, dim=-1)
            nxt = int(torch.multinomial(probs, 1).item())
            gen_tokens.append(nxt)
            if nxt in (0, 50256): break
    gen_text = tokenizer.decode(gen_tokens[len(tokens):])
    gen_text = gen_text.split("<|im_end|>")[0].split("<|endoftext|>")[0].strip()
    return gen_text

# Test on a core logic anchor
test_prompt = "If all humans are mortal and Socrates is human, is Socrates mortal? Explain."
print(f"User: {test_prompt}\n")
response = ask_quillan(test_prompt, max_tokens=100, temperature=0.5)
print(f"Quillan:\n{response}")
# Extra probes
for p in ["What is 2+2?", "Explain BitNet 1.58b in one sentence."]:
    print(f"\nUser: {p}")
    print(f"Quillan: {ask_quillan(p, max_tokens=60, temperature=0.7)}")


In [ ]:
# [Cell 6] Plot Loss Curve
try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(10,4))
    plt.plot(step_losses, label='train step loss')
    plt.axhline(best_loss, color='green', linestyle='--', label=f'best {best_loss:.3f}')
    plt.axhline(3.0, color='red', linestyle=':', label='target sub-3.0')
    plt.xlabel('step in block')
    plt.ylabel('loss')
    plt.legend()
    plt.title('Quillan v5.4-ONI Training Loss')
    plt.show()
except Exception as e:
    print(f"Plot skipped: {e}")
    print(step_losses[:10], "...", step_losses[-10:])


### 🎯 Next Steps:
- Re-run **Cell 4** with more steps whenever you want to train another chunk (e.g., 500 steps).
- Current best is at `checkpoints/checkpoints_sft/quillan_frontier_v2_best.pt` (step 432, loss 6.70).
- Target is **sub-3.0**: expect ~1500-2500 steps on this 3.5k anchor set (effBatch 8 = ~4 epochs per 500 steps).
- On this GTX 1050 machine, **CPU training is forced** (CC 6.1 unsupported). Use Colab T4/A100 for GPU speedup (10-20x).
- Modify `test_prompt` in **Cell 5** to test any question interactively.
